# Primeiro teste de modelo

In [1]:
import sys
!{sys.executable} -m pip install pandas tensorflow scikit-learn


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# 1. Carregar o dataset
data = pd.read_csv('C:/Users/progr/OneDrive/Documentos/Miguel/PROJETOS/Rainha_de_Saba/01_data/processed/bitcoin_processed.csv')  # Substitua pelo caminho correto do seu arquivo
data

ImportError: Traceback (most recent call last):
  File "C:\Users\progr\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Uma rotina de inicialização da biblioteca de vínculo dinâmico (DLL) falhou.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
# 2. Selecionar a coluna de preços (ajuste conforme necessário)
prices = data['Último'].values  # Aqui usamos o preço de fechamento, mas pode incluir outros indicadores

# 3. Normalizar os dados para o intervalo [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices.reshape(-1, 1))

# 4. Criar sequências de dados (Janela de tempo)
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:i + time_step])
        y.append(data[i + time_step])
    return np.array(X), np.array(y)

time_step = 60  # Usar 60 dias para prever o próximo dia
X, y = create_sequences(prices_scaled, time_step)

# 5. Dividir os dados em treino e teste
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [ ]:
# 6. Modelar a LSTM
model = Sequential()

# Camada LSTM com 50 unidades
model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])))

# Camada densa (fully connected) para prever o valor de saída
model.add(Dense(units=1))

# 7. Compilar o modelo
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# 8. Treinar o modelo
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# 9. Previsão
predictions = model.predict(X_test)

# 10. Inverter a normalização para obter os valores reais
predictions = scaler.inverse_transform(predictions)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
# 11. Exibir os resultados
print("Previsões:", predictions[:10])
print("Valores reais:", y_test[:10])